In [96]:
import os
import numpy
import matplotlib.pyplot as plt
import pandas as pd

from PIL import Image
from IPython.display import display
from typing import Optional, Dict, Any
from typing_extensions import TypedDict
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool, StructuredTool
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import START, END, StateGraph

from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

from colorama import init, Fore, Style, Back
init() 

load_dotenv()

True

In [2]:
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY  = os.getenv("TAVILY_API_KEY")

## Basic Prompt

In [3]:
# Example prompt for the math tutor
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math tutor. Explain step-by-step and use simple language."),
    ("human", "{input}")
])

# HOOK LLM Server

In [4]:
llm     = ChatOpenAI(
    api_key         = OPENAI_API_KEY
    , model         = "gpt-4o-mini-2024-07-18"
    , temperature   = 0.35
)

## Parsing LLM with Prompt as Chain

In [5]:
chain   = prompt | llm

In [6]:
chain.invoke({"input": "Hi, Could u tell me what is the probability?"})

AIMessage(content="Of course! Let's break down what probability is in a simple way.\n\n### What is Probability?\n\n**Probability** is a way to measure how likely something is to happen. It gives us a number between 0 and 1 (or sometimes as a percentage) that tells us how likely an event is.\n\n- **0** means the event will not happen at all.\n- **1** means the event will definitely happen.\n- A probability of **0.5** means there is an equal chance of the event happening or not happening.\n\n### How to Calculate Probability\n\nTo calculate the probability of an event, you can use this simple formula:\n\n\\[\n\\text{Probability (P)} = \\frac{\\text{Number of favorable outcomes}}{\\text{Total number of possible outcomes}}\n\\]\n\n#### Step-by-Step Example\n\nLet’s say you have a simple example: rolling a six-sided die.\n\n1. **Identify the event**: Let's say we want to find the probability of rolling a 4.\n   \n2. **Count the favorable outcomes**: There is **1** way to roll a 4 (just rolli

## LLM as Stream

In [7]:
llm     = ChatOpenAI(
    api_key         = OPENAI_API_KEY
    , model         = "gpt-4o-mini-2024-07-18"
    , temperature   = 0.35
    , streaming     = True
)

In [8]:
# Helper FUncs

def display_graph_image_inline(
    image_path      : str
) -> None:
    img     = Image.open(image_path)
    display(img)

In [9]:
chain   = prompt | llm

In [10]:
for chunk in chain.stream({"input": "Hi, Could u tell me what is the probability?"}):
    print(chunk.content, end="", flush=True)

Sure! Probability is a way to measure how likely something is to happen. It helps us understand the chance of different outcomes in uncertain situations.

Here’s a simple way to think about it:

1. **Understanding Outcomes**: An outcome is a possible result of an event. For example, if you flip a coin, the outcomes are "heads" or "tails."

2. **Total Outcomes**: To find the probability, you first need to know how many total outcomes there are. In the coin example, there are 2 total outcomes (heads and tails).

3. **Favorable Outcomes**: Next, you need to identify the outcomes that you are interested in. If you want to know the probability of getting heads, there is 1 favorable outcome (heads).

4. **Calculating Probability**: The probability is calculated using the formula:

   \[
   \text{Probability} = \frac{\text{Number of Favorable Outcomes}}{\text{Total Number of Outcomes}}
   \]

   In our coin example, the probability of getting heads would be:

   \[
   \text{Probability of hea

## Structured Output and Tool Usage 

In [11]:
# Define the output type model
class ProbabilityCalculation(BaseModel):
    problem_type            : str               = Field(description="Type of probability problem (e.g., binomial, normal)")
    probability_value       : float             = Field(description="The calculated probability (between 0 and 1)")
    mean                    : Optional[float]   = Field(default=None, description="Mean of the distribution")
    std_deviation           : Optional[float]   = Field(default=None, description="Standard deviation")
    formula_used            : str               = Field(description="The formula used for calculation")
    explanation             : str               = Field(description="Step-by-step explanation")


# New prompt    
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert in probability calculations. 
    When given a probability problem, calculate and explain:
    - The exact probability value
    - Mean and standard deviation (if applicable)
    - The formula used
    - Step-by-step explanation
    
    Format your response according to the structured schema."""),
    ("human", "{input}")
])

In [12]:
llm = ChatOpenAI(
    api_key         = OPENAI_API_KEY
    , model         = "gpt-4o-mini-2024-07-18"
    , temperature   = 0.35
    , streaming     = True
)

chain   = prompt | llm.with_structured_output(ProbabilityCalculation)

In [13]:
response = chain.invoke({
    "input": "What's the probability of getting exactly 3 heads in 5 coin flips?"
})

In [14]:
print(f"Problem Type: {response.problem_type}")
print(f"Probability: {response.probability_value}")
print(f"Mean: {response.mean}")
print(f"Standard Deviation: {response.std_deviation}")
print(f"Formula: {response.formula_used}")
print(f"Explanation: {response.explanation}")

Problem Type: binomial
Probability: 0.3125
Mean: 2.5
Standard Deviation: 1.118033988749895
Formula: P(X = k) = C(n, k) * p^k * (1-p)^(n-k)
Explanation: To find the probability of getting exactly 3 heads in 5 coin flips, we can use the binomial probability formula:

1. Identify the parameters:
   - n = number of trials (coin flips) = 5
   - k = number of successes (heads) = 3
   - p = probability of success on a single trial (getting heads) = 0.5

2. Calculate the binomial coefficient C(n, k):
   C(5, 3) = 5! / (3! * (5-3)!) = 5! / (3! * 2!) = (5 * 4) / (2 * 1) = 10

3. Calculate the probability:
   P(X = 3) = C(5, 3) * (0.5)^3 * (0.5)^(5-3)
   = 10 * (0.5)^3 * (0.5)^2
   = 10 * (0.5)^5
   = 10 * (1/32)
   = 10/32 = 0.3125

4. Mean and Standard Deviation:
   - Mean (μ) of a binomial distribution is given by μ = n * p = 5 * 0.5 = 2.5
   - Standard Deviation (σ) is given by σ = sqrt(n * p * (1-p)) = sqrt(5 * 0.5 * 0.5) = sqrt(1.25) ≈ 1.118.

Thus, the probability of getting exactly 3 head

In [15]:
for chunk in chain.stream({"input": "What's the probability of getting exactly 3 heads in 5 coin flips?"}):
    if hasattr(chunk, 'explanation') and chunk.explanation:
        print(chunk.explanation, end="", flush=True)

To find the probability of getting exactly 3 heads in 5 coin flips, we can use the binomial probability formula. Here, we have:

- n = 5 (the number of trials or coin flips)
- k = 3 (the number of successful outcomes, which is getting heads)
- p = 0.5 (the probability of getting heads in a single flip)

1. First, we calculate the binomial coefficient (n choose k), which is given by the formula:
   (n choose k) = n! / (k! * (n-k)!)
   For our case:
   (5 choose 3) = 5! / (3! * (5-3)!) = 5! / (3! * 2!) = (5 * 4) / (2 * 1) = 10

2. Next, we calculate p^k and (1-p)^(n-k):
   p^k = (0.5)^3 = 0.125
   (1-p)^(n-k) = (0.5)^(5-3) = (0.5)^2 = 0.25

3. Now we can plug these values into the binomial probability formula:
   P(X = 3) = (5 choose 3) * (0.5)^3 * (0.5)^(5-3)
   P(X = 3) = 10 * 0.125 * 0.25 = 10 * 0.03125 = 0.3125

Thus, the probability of getting exactly 3 heads in 5 coin flips is 0.3125.

4. To find the mean and standard deviation:
   - Mean (μ) of a binomial distribution is given by 

In [16]:
# Tool Usage case
search_tool = TavilySearchResults(max_results=3)

prompt      = ChatPromptTemplate.from_messages([
    ("system", """You are a statistician who breaks down problems step by step.
    - Use search when you need current data or recent studies
    - Apply statistical methods and formulas appropriately
    - Clearly explain your reasoning
    - Cite any sources you find"""),
    ("human", "{input}")
])

llm = ChatOpenAI(
    api_key         = OPENAI_API_KEY
    , model         = "gpt-4o-mini-2024-07-18"
    , temperature   = 0.35
    , streaming     = True
)

chain       = prompt | llm.bind_tools([search_tool])

response    = chain.invoke({"input": "How can we determine if there's significant correlation between study hours and exam scores?"})

print(response.content)
print("\nTool calls made:")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

To determine if there's a significant correlation between study hours and exam scores, we can follow these steps:

### Step 1: Collect Data
Gather data on the number of study hours and the corresponding exam scores for a group of students. This data can be organized in a table where one column represents study hours and the other represents exam scores.

### Step 2: Visualize the Data
Create a scatter plot with study hours on the x-axis and exam scores on the y-axis. This visualization helps to see if there is a linear relationship between the two variables.

### Step 3: Calculate the Correlation Coefficient
The correlation coefficient (usually represented as \( r \)) quantifies the degree of linear relationship between the two variables. The formula for Pearson's correlation coefficient is:

\[
r = \frac{n(\sum xy) - (\sum x)(\sum y)}{\sqrt{[n\sum x^2 - (\sum x)^2][n\sum y^2 - (\sum y)^2]}}
\]

Where:
- \( n \) = number of pairs of scores
- \( x \) = study hours
- \( y \) = exam score

## Agent

In [40]:
def create_sample_pokemon_data():
    data = {
        'Name': ['Bulbasaur', 'Charmander', 'Squirtle', 'Pidgey', 'Rattata', 
                 'Spearow', 'Ekans', 'Pikachu', 'Sandshrew', 'Nidoran♀'],
        'Type 1': ['Grass', 'Fire', 'Water', 'Normal', 'Normal', 
                   'Normal', 'Poison', 'Electric', 'Ground', 'Poison'],
        'Type 2': ['Poison', None, None, 'Flying', None, 
                   'Flying', None, None, None, None],
        'HP': [45, 39, 44, 40, 30, 40, 35, 35, 50, 55],
        'Attack': [49, 52, 48, 45, 56, 60, 60, 55, 75, 47],
        'Defense': [49, 43, 65, 40, 35, 30, 44, 40, 85, 52],
        'Speed': [45, 65, 43, 56, 72, 70, 55, 90, 40, 41]
    }

    return pd.DataFrame(data)

In [41]:
pokemon_df  = create_sample_pokemon_data()

In [42]:
# Defining desire output
class StatisticalSummary(BaseModel):
    mean        : float = Field(description="Mean value")
    median      : float = Field(description="Median value")
    std_dev     : float = Field(description="Standard deviation")
    min_value   : float = Field(description="Minimum value")
    max_value   : float = Field(description="Maximum value")
    count       : int   = Field(description="Count of non-null values")

class CorrelationAnalysis(BaseModel):
    correlation_coefficient : float = Field(description="Pearson correlation coefficient")
    correlation_strength    : str   = Field(description="Description of correlation strength")
    p_value                 : Optional[float] = Field(default=None, description="Statistica significance p-value")
    interpretation          : str   = Field(description="Interpretation of the correlation")

class TypeComparison(BaseModel):
    type1_mean              : float = Field(description="Mean value for type 1")
    type2_mean              : float = Field(description="Mean value for type 2")
    difference              : float = Field(description="Difference between types")
    statistical_significance: bool  = Field(description="Whether the difference is statistically significant")
    explanation             : str   = Field(description="Interpretation of the comparison")

class AnalysisResult(BaseModel):
    analysis_type           : str = Field(description="Type of analysis performed")
    summary                 : str = Field(description="High-level summary of findings")
    detailed_results        : Dict[str, Any] = Field(description="Detailed analysis results")
    visualization_created   : bool = Field(description="Whether a visualization was created")
    visualization_path      : Optional[str] = Field(default=None, description="Path to saved visualization")

In [122]:
# Define needed tools
@tool
def calculate_basic_statistics(column: str) -> StatisticalSummary:
    """Calculate basic statistics for a given column in the dataframe"""
    
    data = pokemon_df[column].dropna()
    
    return StatisticalSummary(
        mean            = float(data.mean())
        , median        = float(data.median())
        , std_dev       = float(data.std())
        , min_value     = float(data.min())
        , max_value     = float(data.max())
        , count         = len(data)
    )


@tool
def compare_pokemon_types(
    attribute   : str
    , type1     : str
    , type2     : str
) -> TypeComparison:
    """Compare a statistical attribute between two Pokemon types"""

    type1_data  = pokemon_df[pokemon_df['Type 1'] == type1][attribute].dropna()
    type2_data  = pokemon_df[pokemon_df['Type 1'] == type2][attribute].dropna()

    if len(type1_data) < 2 or len(type2_data) < 2:
        explanation = (
            f"Insufficient data for comparison. {type1}: {len(type1_data)} samples, "
            f"{type2}: {len(type2_data)} samples"
        )
        return TypeComparison(
            type1_mean                  = float(type1_data.mean()) if len(type1_data) > 0 else 0.0
            , type2_mean                = float(type2_data.mean()) if len(type2_data) > 0 else 0.0
            , difference                = 0.0
            , statistical_significance  = False
            , explanation               = explanation
        )

    type1_mean  = type1_data.mean()
    type2_mean  = type2_data.mean()
    difference  = type1_mean - type2_mean

    try:
        from scipy import stats
        _, p_value      = stats.ttest_ind(type1_data, type2_data)
        is_significant  = p_value < 0.05
        if p_value != p_value:  
            is_significant  = False
            p_value         = float('nan')
    except:
        p_value         = float('nan')
        is_significant  = False

    import numpy as np
    return TypeComparison(
        type1_mean                  = float(type1_mean)
        , type2_mean                = float(type2_mean)
        , difference                = float(difference)
        , statistical_significance  = is_significant
        , explanation               = (
            f"The {attribute} difference between {type1} ({type1_mean:.2f}) and {type2} ({type2_mean:.2f}) is "
            f"{'significant' if is_significant else 'not significant'}"
            f"{' with p-value {:.4f}'.format(p_value) if not np.isnan(p_value) else ' (p-value not calculated)'}"
        )
    )



@tool
def calculate_correlation(column1: str, column2: str) -> CorrelationAnalysis:
    """Calculate correlation between two columns"""
    data1 = pokemon_df[column1].dropna()
    data2 = pokemon_df[column2].dropna()
    
    valid_indices   = data1.index.intersection(data2.index)
    data1 = data1.loc[valid_indices]
    data2 = data2.loc[valid_indices]
    
    correlation     = data1.corr(data2)
    
    # Determine correlation strength
    abs_corr = abs(correlation)
    if abs_corr > 0.8:
        strength = "Very Strong"
    elif abs_corr > 0.6:
        strength = "Strong"
    elif abs_corr > 0.4:
        strength = "Moderate"
    elif abs_corr > 0.2:
        strength = "Weak"
    else:
        strength = "Very Weak"
    
    # Calculate p-value
    from scipy import stats
    _, p_value = stats.pearsonr(data1, data2)
    
    return CorrelationAnalysis(
        correlation_coefficient = float(correlation)
        , correlation_strength  = strength
        , p_value               = float(p_value)
        , interpretation        = f"There is a {strength.lower()} {'positive' if correlation > 0 else 'negative'} correlation (r={correlation:.3f}) between {column1} and {column2}"
    )

@tool
def create_visualization(plot_type: str, x_column: str = None, y_column: str = None, group_by: str = None) -> str:
    """Create statistical visualizations"""
    plt.figure(figsize=(10, 6))
    
    if plot_type    == "histogram":
        pokemon_df[x_column].hist(bins=20)
        plt.title(f"Distribution of {x_column}")
        plt.xlabel(x_column)
        plt.ylabel("Frequency")
    
    elif plot_type  == "scatter":
        plt.scatter(pokemon_df[x_column], pokemon_df[y_column])
        plt.title(f"{x_column} vs {y_column}")
        plt.xlabel(x_column)
        plt.ylabel(y_column)
    
    elif plot_type  == "boxplot":
        if group_by:
            pokemon_df.boxplot(column=x_column, by=group_by)
            plt.title(f"{x_column} by {group_by}")
        else:
            pokemon_df[x_column].boxplot()
            plt.title(f"Distribution of {x_column}")
    
    elif plot_type == "bar":
        if group_by:
            pokemon_df.groupby(group_by)[x_column].mean().plot(kind='bar')
            plt.title(f"Average {x_column} by {group_by}")
        else:
            pokemon_df[x_column].value_counts().plot(kind='bar')
            plt.title(f"Count of {x_column}")
    
    filename = f"pokemon_analysis_{plot_type}.png"
    plt.savefig(filename)
    plt.close()
    
    return filename

@tool
def find_strongest_pokemon(
    attribute   : str = "Attack"
) -> Dict[str, Any]:
    """Find the Pokemon with the highest value for a given attribute"""

    if attribute == "Total":
        pokemon_df['TotalPower']    = pokemon_df[['HP', 'Attack', 'Defense', 'Speed']].sum(axis=1)
        max_value                   = pokemon_df['TotalPower'].max()
        strongest_pokemon           = pokemon_df[pokemon_df['TotalPower'] == max_value].iloc[0]
        attribute_value             = max_value
        pokemon_df.drop('TotalPower', axis=1, inplace=True)
    else:
        if attribute not in pokemon_df.columns:
            raise ValueError(f"Attribute {attribute} not found in dataframe")
        max_value                   = pokemon_df[attribute].max()
        strongest_pokemon           = pokemon_df[pokemon_df[attribute] == max_value].iloc[0]
        attribute_value             = max_value

    power_score = strongest_pokemon[['HP', 'Attack', 'Defense', 'Speed']].sum()

    return {
        "name"      : strongest_pokemon['Name']
        , "attribute"   : attribute
        , "value"       : float(attribute_value)
        , "stats"       : {
            "HP"            : float(strongest_pokemon['HP'])
            , "Attack"      : float(strongest_pokemon['Attack'])
            , "Defense"     : float(strongest_pokemon['Defense'])
            , "Speed"       : float(strongest_pokemon['Speed'])
            , "Total Power" : float(power_score)
        }
        , "type1"     : strongest_pokemon['Type 1']
        , "type2"     : strongest_pokemon['Type 2'] if pd.notna(strongest_pokemon['Type 2']) else None
    }

In [123]:
# Define Agent State
class AgentState(TypedDict):
    user_query          : str
    analysis_type       : str
    summary             : str
    analysis_parameters : Dict[str, Any]
    results             : Dict[str, Any]
    visualization_path  : Optional[str]

In [124]:
def interpret_query_node(
    state           : AgentState
) -> Dict[str, Any]:
    """Interpret the user's query using LLM to determine analysis type and parameters"""

    llm = ChatOpenAI(
        api_key     = OPENAI_API_KEY
        , model     = "gpt-4o-mini"
        , temperature = 0.1
    )

    interpretation_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a query interpreter for Pokemon statistical analysis. 
        Analyze the user's query and determine:
        1. The analysis type (basic_stats, correlation, type_comparison, find_strongest, or visualization)
        2. The specific parameters needed

        Available analysis types:
        - basic_stats: For general statistics about a single attribute
        - correlation: For finding relationships between two attributes
        - type_comparison: For comparing Pokemon types
        - find_strongest: For finding the most powerful Pokemon
        - visualization: For creating plots and charts

        Available attributes: HP, Attack, Defense, Speed
        Available Pokemon types: Fire, Water, Grass, Normal, Electric, Poison, Ground

        Respond with just the analysis type and parameters in this format:
        analysis_type: <type>
        attribute: <attribute if needed>
        column1: <column if needed>
        column2: <column if needed>
        type1: <type if needed>
        type2: <type if needed>
        plot_type: <plot type if needed>
        x_column: <column if needed>
        y_column: <column if needed>
        """)
        , ("human", "{query}")
    ])

    response        = llm.invoke(interpretation_prompt.format(query=state["user_query"]))

    lines           = response.content.strip().split('\n')
    analysis_type   = "basic_stats"
    parameters      = {}

    for line in lines:
        if ':' in line:
            key, value = line.split(':', 1)
            key   = key.strip()
            value = value.strip()
            if key == 'analysis_type':
                analysis_type = value
            elif value and value != '<none>':
                parameters[key] = value

    return {
        "analysis_type"        : analysis_type
        , "analysis_parameters": parameters
    }


In [125]:
def execute_analysis_node(
    state           : AgentState
) -> Dict[str, Any]:
    """Execute the appropriate analysis based on the interpreted query"""

    analysis_type        = state["analysis_type"]
    parameters           = state["analysis_parameters"]
    results              = {}
    visualization_path   = None

    if analysis_type == "basic_stats":
        column      = parameters.get("column", "Attack")
        result      = calculate_basic_statistics.invoke({
            "column"    : column
        })
        results["statistics"] = result.model_dump()  

    elif analysis_type == "find_strongest":
        attribute   = parameters.get("attribute", "Total")
        result      = find_strongest_pokemon.invoke({"attribute": attribute})
        results["strongest_pokemon"] = result

    elif analysis_type == "correlation":
        col1        = parameters.get("column1", "Attack")
        col2        = parameters.get("column2", "Defense")
        result      = calculate_correlation.invoke({
            "column1"   : col1
            , "column2" : col2
        })
        results["correlation"] = result.model_dump()  

    elif analysis_type == "type_comparison":
        attribute   = parameters.get("attribute", "Attack")
        type1       = parameters.get("type1", "Fire")
        type2       = parameters.get("type2", "Water")
        result      = compare_pokemon_types.invoke({
            "attribute" : attribute
            , "type1"   : type1
            , "type2"   : type2
        })
        results["type_comparison"] = result.model_dump()  

    elif analysis_type == "visualization":
        plot_type   = parameters.get("plot_type", "histogram")
        x_column    = parameters.get("x_column", "Attack")
        y_column    = parameters.get("y_column")
        group_by    = parameters.get("group_by")
        
        call_params = {"plot_type": plot_type, "x_column": x_column}
        if y_column is not None:
            call_params["y_column"] = y_column
        if group_by is not None:
            call_params["group_by"] = group_by
        
        result                      = create_visualization.invoke(call_params)
        visualization_path          = result
        results["visualization"]    = result

    return {
        "results"               : results
        , "visualization_path"  : visualization_path
    }

In [126]:
def generate_summary_node(
    state           : AgentState
) -> Dict[str, Any]:
    """Generate a summary of the analysis results using LLM"""

    llm = ChatOpenAI(
        api_key     = OPENAI_API_KEY
        , model     = "gpt-4o-mini-2024-07-18"
        , temperature = 0.1
        , streaming = True
    )

    analysis_type    = state["analysis_type"]
    results          = state["results"]
    results_text     = ""
    if analysis_type == "basic_stats":
        stats           = results.get("statistics", {})
        results_text    = (
            f"Column: {state['analysis_parameters'].get('column', 'unknown')}\n"
            f"Mean: {stats.get('mean', 0):.2f}\n"
            f"Median: {stats.get('median', 0):.2f}\n"
            f"Standard Deviation: {stats.get('std_dev', 0):.2f}\n"
            f"Min: {stats.get('min_value', 0):.2f}\n"
            f"Max: {stats.get('max_value', 0):.2f}\n"
            f"Count: {stats.get('count', 0)}"
        )

    elif analysis_type == "correlation":
        corr        = results.get("correlation", {})
        results_text = (
            f"Correlation between {state['analysis_parameters'].get('column1')} and {state['analysis_parameters'].get('column2')}:\n"
            f"Coefficient: {corr.get('correlation_coefficient', 0):.3f}\n"
            f"Strength: {corr.get('correlation_strength', 'Unknown')}\n"
            f"P-value: {corr.get('p_value', 0):.4f}\n"
            f"Interpretation: {corr.get('interpretation', '')}"
        )

    elif analysis_type == "type_comparison":
        comp        = results.get("type_comparison", {})
        results_text = (
            f"Comparison of {state['analysis_parameters'].get('attribute')} between {state['analysis_parameters'].get('type1')} and {state['analysis_parameters'].get('type2')}:\n"
            f"Type 1 mean: {comp.get('type1_mean', 0):.2f}\n"
            f"Type 2 mean: {comp.get('type2_mean', 0):.2f}\n"
            f"Difference: {comp.get('difference', 0):.2f}\n"
            f"Statistically significant: {comp.get('statistical_significance', False)}\n"
            f"Explanation: {comp.get('explanation', '')}"
        )

    elif analysis_type == "visualization":
        results_text = (
            f"Visualization created: {state.get('visualization_path', 'No path available')}"
        )

    elif analysis_type == "find_strongest":
        strongest = results.get("strongest_pokemon", {})
        results_text = (
            f"The strongest Pokemon based on {strongest.get('attribute', 'overall power')}:\n"
            f"Name: {strongest.get('name', 'Unknown')}\n"
            f"Type: {strongest.get('type1', 'Unknown')}"
            f"{' / ' + strongest['type2'] if strongest.get('type2') else ''}\n"
            f"Stats:\n"
            f"  HP: {strongest.get('stats', {}).get('HP', 0):.0f}\n"
            f"  Attack: {strongest.get('stats', {}).get('Attack', 0):.0f}\n"
            f"  Defense: {strongest.get('stats', {}).get('Defense', 0):.0f}\n"
            f"  Speed: {strongest.get('stats', {}).get('Speed', 0):.0f}\n"
            f"  Total Power: {strongest.get('stats', {}).get('Total Power', 0):.0f}"
        )
    
    prompt = (
        f"Please provide a clear, natural language summary of the following statistical analysis results:\n\n"
        f"Analysis Type: {analysis_type}\n"
        f"User Query: {state['user_query']}\n\n"
        f"Results:\n"
        f"{results_text}\n\n"
        f"Create a concise but informative summary that explains the findings in simple terms."
    )

    response    = llm.invoke(prompt)
    summary     = response.content

    # print(f"Debuggint response {response.content}")

    return {
        "summary"   : summary
    }

In [ ]:
# Create Agent
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("interpret_query", interpret_query_node)
workflow.add_node("execute_analysis", execute_analysis_node)
workflow.add_node("generate_summary", generate_summary_node)

# Define edges
workflow.add_edge(START, "interpret_query")
workflow.add_edge("interpret_query", "execute_analysis")
workflow.add_edge("execute_analysis", "generate_summary")
workflow.add_edge("generate_summary", END)

# Compile the graph
agent   = workflow.compile()

try:

    graph_image = agent.get_graph().draw_mermaid_png()
    with open("pokemon_agent_workflow.png", "wb") as f:
        f.write(graph_image)
    
except Exception as e:
    print(f"Error saving graph: {e}")

Error saving graph: 'module' object is not callable


In [ ]:
test_queries = [
    "What are the basic statistics for Pokemon Attack values?",
    "Is there a correlation between HP and Defense?",
    "Compare the average speed between Fire and Water Pokemon",
    "Create a histogram showing the distribution of Defense stats",
    "What is the most power full Pokemon, Can u Give me the Name of it ?"
]

In [129]:
import time
import textwrap

for query in test_queries:
    print(f"\n{Back.CYAN}{Fore.BLACK}{'='*52}")
    print(f"📊 Query: {query}")
    print(f"{'='*52}{Style.RESET_ALL}")

    initial_state = {
        "user_query"            : query
        , "analysis_type"       : ""
        , "analysis_parameters" : {}
        , "results"             : {}
        , "visualization_path"  : None
        , "summary"             : ""
    }

    print(f"\n{Fore.GREEN}🔄 Processing...{Style.RESET_ALL}")
    for chunk in agent.stream(initial_state):
        if "interpret_query" in chunk:
            print(f"{Fore.YELLOW}🤔 Interpreting query: '{query}'{Style.RESET_ALL}")

        if "execute_analysis" in chunk:
            print(f"{Fore.BLUE}📈 Executing analysis...{Style.RESET_ALL}")

        if "generate_summary" in chunk:
            print(f"{Fore.MAGENTA}✍️ Generating summary...{Style.RESET_ALL}")

    result = agent.invoke(initial_state)

    print(f"\n{Back.GREEN}{Fore.BLACK}{'✅ RESULTS'}".center(54) + f"{Style.RESET_ALL}")
    print(f"{Fore.CYAN}Analysis Type: {Fore.YELLOW}{result['analysis_type'].upper()}{Style.RESET_ALL}")
    print(f"\n{Fore.CYAN}📝 Summary:{Style.RESET_ALL}")

    summary = result.get('summary', 'No summary available')

    wrapped_lines = []
    for line in summary.split('\n'):
        if line.strip():
            wrapped_lines.extend(textwrap.wrap(line, width=65))
        else:
            wrapped_lines.append('')

    for line_idx, line in enumerate(wrapped_lines):
        if not line.strip():
            print()
            continue

        if line.strip().endswith(':'):
            color = Fore.YELLOW
        elif any(char.isdigit() for char in line):
            color = Fore.GREEN
        else:
            color = Fore.WHITE

        for char in line:
            print(f"{color}{char}{Style.RESET_ALL}", end='', flush=True)
            time.sleep(0.015)
        print()  

    if result.get('visualization_path'):
        print(f"\n{Fore.CYAN}📊 Visualization saved to: {Fore.YELLOW}{result['visualization_path']}{Style.RESET_ALL}")

    print(f"\n{Back.MAGENTA}{Fore.BLACK}{'-'*52}")
    print(f"✨ Done!")
    print(f"{'-'*52}{Style.RESET_ALL}")

    time.sleep(0.5)


📊 Query: What is the most power full Pokemon, Can u Give me the Name of it ?

🔄 Processing...
🤔 Interpreting query: 'What is the most power full Pokemon, Can u Give me the Name of it ?'
📈 Executing analysis...
✍️ Generating summary...
                 
✅ RESULTS                 
Analysis Type: FIND_STRONGEST

📝 Summary:
The analysis identified Sandshrew as the strongest Pokémon based
on its total power score. Sandshrew is a Ground-type Pokémon with
the following stats: 50 HP, 75 Attack, 85 Defense, and 40 Speed,
resulting in a total power of 250.

----------------------------------------------------
✨ Done!
----------------------------------------------------


In [136]:
def interpret_query_node(
    state           : AgentState
) -> Dict[str, Any]:
    """Interpret the user's query using LLM to determine analysis type and parameters"""

    llm = ChatOpenAI(
        api_key     = OPENAI_API_KEY
        , model     = "gpt-4o-mini"
        , temperature = 0.1
    )

    interpretation_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a query interpreter for Pokemon statistical analysis. 
        Analyze the user's query and determine:
        1. The analysis type (basic_stats, correlation, type_comparison, find_strongest, or visualization)
        2. The specific parameters needed

        Available analysis types:
        - basic_stats: For general statistics about a single attribute
        - correlation: For finding relationships between two attributes
        - type_comparison: For comparing Pokemon types
        - find_strongest: For finding the most powerful Pokemon
        - visualization: For creating plots and charts

        Available attributes: HP, Attack, Defense, Speed
        Available Pokemon types: Fire, Water, Grass, Normal, Electric, Poison, Ground

        Respond with just the analysis type and parameters in this format:
        analysis_type: <type>
        attribute: <attribute if needed>
        column1: <column if needed>
        column2: <column if needed>
        type1: <type if needed>
        type2: <type if needed>
        plot_type: <plot type if needed>
        x_column: <column if needed>
        y_column: <column if needed>
        """)
        , ("human", "{query}")
    ])

    response        = llm.invoke(interpretation_prompt.format(query=state["user_query"]))

    lines           = response.content.strip().split('\n')
    analysis_type   = "basic_stats"
    parameters      = {}

    for line in lines:
        if ':' in line:
            key, value = line.split(':', 1)
            key   = key.strip()
            value = value.strip()
            if key == 'analysis_type':
                analysis_type = value
            elif value and value != '<none>':
                parameters[key] = value

    return {
        "analysis_type"        : analysis_type
        , "analysis_parameters": parameters
    }

def decision_node(
    state           : AgentState
) -> Dict[str, Any]:
    """Make decisions about how to handle different queries"""

    llm = ChatOpenAI(
        api_key     = OPENAI_API_KEY
        , model     = "gpt-4o-mini"
        , temperature = 0.1
    )

    decision_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a decision maker for Pokemon analysis. 
        Based on the user's query and analysis type, determine if the query needs:
        1. A simple single analysis
        2. Multiple analyses to fully answer the question
        3. A narrative explanation combining multiple insights

        Respond with:
        decision: <single/multiple/narrative>
        additional_analyses: <comma-separated list if multiple>
        explanation_focus: <what aspects to emphasize>
        """)
        , ("human", """User Query: {query}
        Analysis Type: {analysis_type}
        Parameters: {parameters}""")
    ])

    response = llm.invoke(decision_prompt.format(
        query           = state["user_query"]
        , analysis_type = state["analysis_type"]
        , parameters    = state["analysis_parameters"]
    ))

    lines               = response.content.strip().split('\n')
    decision            = "single"
    additional_analyses = []
    explanation_focus   = ""

    for line in lines:
        if ':' in line:
            key, value = line.split(':', 1)
            key   = key.strip()
            value = value.strip()
            if key == 'decision':
                decision = value
            elif key == 'additional_analyses' and value:
                additional_analyses = [a.strip() for a in value.split(',')]
            elif key == 'explanation_focus':
                explanation_focus = value

    return {
        "decision"              : decision
        , "additional_analyses" : additional_analyses
        , "explanation_focus"   : explanation_focus
    }
workflow = StateGraph(AgentState)

workflow.add_node("interpret_query", interpret_query_node)
workflow.add_node("decision", decision_node)
workflow.add_node("execute_analysis", execute_analysis_node)
workflow.add_node("generate_summary", generate_summary_node)

workflow.add_edge(START, "interpret_query")
workflow.add_edge("interpret_query", "decision")

def should_continue(state):
    decision = state.get("decision", "single")
    if decision == "multiple":
        return "execute_multiple"
    elif decision == "narrative":
        return "execute_narrative"
    else:
        return "execute_analysis"

workflow.add_conditional_edges(
    "decision",
    should_continue,
    {
        "execute_analysis": "execute_analysis",
        "execute_multiple": "execute_analysis",
        "execute_narrative": "execute_analysis"
    }
)

workflow.add_edge("execute_analysis", "generate_summary")
workflow.add_edge("generate_summary", END)

agent = workflow.compile()

In [137]:

try:

    graph_image = agent.get_graph().draw_mermaid_png()
    with open("pokemon_agent_workflow_daptive.png", "wb") as f:
        f.write(graph_image)
    
except Exception as e:
    print(f"Error saving graph: {e}")

In [132]:
import time
import textwrap

for query in test_queries:
    print(f"\n{Back.CYAN}{Fore.BLACK}{'='*52}")
    print(f"📊 Query: {query}")
    print(f"{'='*52}{Style.RESET_ALL}")

    initial_state = {
        "user_query"            : query
        , "analysis_type"       : ""
        , "analysis_parameters" : {}
        , "results"             : {}
        , "visualization_path"  : None
        , "summary"             : ""
    }

    print(f"\n{Fore.GREEN}🔄 Processing...{Style.RESET_ALL}")
    for chunk in agent.stream(initial_state):
        if "interpret_query" in chunk:
            print(f"{Fore.YELLOW}🤔 Interpreting query: '{query}'{Style.RESET_ALL}")

        if "execute_analysis" in chunk:
            print(f"{Fore.BLUE}📈 Executing analysis...{Style.RESET_ALL}")

        if "generate_summary" in chunk:
            print(f"{Fore.MAGENTA}✍️ Generating summary...{Style.RESET_ALL}")

    result = agent.invoke(initial_state)

    print(f"\n{Back.GREEN}{Fore.BLACK}{'✅ RESULTS'}".center(54) + f"{Style.RESET_ALL}")
    print(f"{Fore.CYAN}Analysis Type: {Fore.YELLOW}{result['analysis_type'].upper()}{Style.RESET_ALL}")
    print(f"\n{Fore.CYAN}📝 Summary:{Style.RESET_ALL}")

    summary = result.get('summary', 'No summary available')

    wrapped_lines = []
    for line in summary.split('\n'):
        if line.strip():
            wrapped_lines.extend(textwrap.wrap(line, width=65))
        else:
            wrapped_lines.append('')

    for line_idx, line in enumerate(wrapped_lines):
        if not line.strip():
            print()
            continue

        if line.strip().endswith(':'):
            color = Fore.YELLOW
        elif any(char.isdigit() for char in line):
            color = Fore.GREEN
        else:
            color = Fore.WHITE

        for char in line:
            print(f"{color}{char}{Style.RESET_ALL}", end='', flush=True)
            time.sleep(0.015)
        print()  

    if result.get('visualization_path'):
        print(f"\n{Fore.CYAN}📊 Visualization saved to: {Fore.YELLOW}{result['visualization_path']}{Style.RESET_ALL}")

    print(f"\n{Back.MAGENTA}{Fore.BLACK}{'-'*52}")
    print(f"✨ Done!")
    print(f"{'-'*52}{Style.RESET_ALL}")

    time.sleep(0.5)


📊 Query: What is the most power full Pokemon, Can u Give me the Name of it ?

🔄 Processing...
🤔 Interpreting query: 'What is the most power full Pokemon, Can u Give me the Name of it ?'
📈 Executing analysis...
✍️ Generating summary...
                 
✅ RESULTS                 
Analysis Type: FIND_STRONGEST

📝 Summary:
The analysis identified Sandshrew as the strongest Pokémon based
on its total power score. Sandshrew is a Ground-type Pokémon with
the following stats: 50 HP, 75 Attack, 85 Defense, and 40 Speed,
resulting in a total power of 250.

----------------------------------------------------
✨ Done!
----------------------------------------------------
